# Data Information

This notebook goes over information of the three files we will be working with

- Separations: data on the events of employees leaving federal workforce
- Accessions: data on the events employees joining federal workforce
- Employment: various descriptive data on the federal workforce

# Load Data

In [33]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))
from scripts.data_loader import load_r2_data

In [34]:
#  Load dataframes (choose most recent)
s_df = load_r2_data("separations", year=2026, month=2)
a_df = load_r2_data("accessions", year=2026, month=2)
e_df = load_r2_data("employment", year=2026, month=2)


/Users/destin/Projects/federal-doge-study/.venv/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


  loading separations/2026/separations_202602.txt …


/Users/destin/Projects/federal-doge-study/.venv/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


  loading accessions/2026/accessions_202602.txt …


/Users/destin/Projects/federal-doge-study/.venv/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


  loading employment/2026/employment_202602.txt …


In [35]:
# Build code -> descriptor lookup dicts from paired columns in each df
def build_lookups(df):
    lookups = {}
    for col in df.columns:
        if col.endswith("_code"):
            descriptor = col[:-5]  # strip "_code"
            if descriptor in df.columns:
                lookups[col] = (
                    df[[col, descriptor]]
                    .drop_duplicates()
                    .dropna(subset=[col])
                    .set_index(col)[descriptor]
                    .to_dict()
                )
    return lookups

s_lookups = build_lookups(s_df)
a_lookups = build_lookups(a_df)
e_lookups = build_lookups(e_df)

print(f"Separations: {len(s_lookups)} code -> descriptor mappings built")
print(f"Accessions:  {len(a_lookups)} code -> descriptor mappings built")
print(f"Employment:  {len(e_lookups)} code -> descriptor mappings built")

Separations: 22 code -> descriptor mappings built
Accessions:  22 code -> descriptor mappings built
Employment:  21 code -> descriptor mappings built


# Data description

## Separations
Records of federal employees who left government. One row = one separation event.

In [36]:
import pandas as pd

code_cols = [c for c in s_df.columns if c.endswith("_code")]
print(f"Shape: {s_df.shape[0]:,} rows x {s_df.shape[1]} columns")
print(f"\nCode columns ({len(code_cols)}):")
print(code_cols)
print(f"\nAll columns:")
print(s_df.dtypes.to_string())

Shape: 19,170 rows x 68 columns

Code columns (25):
['agency_code', 'agency_subelement_code', 'appointment_type_code', 'bargaining_unit_code', 'consolidated_statistical_area_code', 'core_based_statistical_area_code', 'duty_station_code', 'duty_station_country_code', 'duty_station_county_code', 'duty_station_state_country_territory_code', 'education_level_code', 'flsa_category_code', 'locality_pay_area_code', 'occupational_category_code', 'occupational_group_code', 'occupational_series_code', 'pay_basis_code', 'pay_plan_code', 'personnel_office_identifier_code', 'position_occupied_code', 'separation_category_code', 'step_or_rate_type_code', 'supervisory_status_code', 'tenure_code', 'work_schedule_code']

All columns:
age_bracket                                   object
agency                                        object
agency_code                                   object
agency_subelement                             object
agency_subelement_code                        object
annualize

In [37]:
# Sample — working with code columns
s_df[["separation_category_code", "agency_subelement_code", "age_bracket", "tenure_code", "work_schedule_code"]].head(5)

,separation_category_code,agency_subelement_code,age_bracket,tenure_code,work_schedule_code
0,SD,ST00,55-59,1.0,F
1,SD,AF1M,60-64,1.0,F
2,SD,AF1M,60-64,1.0,F
3,SD,AF1M,60-64,1.0,F
4,SD,AF1M,65 OR MORE,1.0,F


In [38]:
# Key lookup tables for separations
for key in ["separation_category_code", "agency_code", "tenure_code", "work_schedule_code", "education_level_code", "pay_plan_code"]:
    if key in s_lookups:
        print(f"=== {key} ===")
        print(pd.Series(s_lookups[key]).sort_index().to_string())
        print()

=== separation_category_code ===
SA    TRANSFER OUT - INDIVIDUAL TRANSFER
SB          TRANSFER OUT - MASS TRANSFER
SC                                  QUIT
SD                RETIREMENT - VOLUNTARY
SE                RETIREMENT - EARLY OUT
SG                    RETIREMENT - OTHER
SH              REDUCTION IN FORCE (RIF)
SJ      TERMINATION (EXPIRED APPT/OTHER)
SL                      OTHER SEPARATION

=== agency_code ===
AF                 DEPARTMENT OF THE AIR FORCE
AG                   DEPARTMENT OF AGRICULTURE
AH       NAT FOUNDATION ON ARTS AND HUMANITIES
AM           U.S. AGENCY FOR INTERNATIONAL DEV
AN              AFRICAN DEVELOPMENT FOUNDATION
AR                      DEPARTMENT OF THE ARMY
AU           FEDERAL LABOR RELATIONS AUTHORITY
BF     DEFENSE NUCLEAR FACILITIES SAFETY BOARD
BG        PENSION BENEFIT GUARANTY CORPORATION
BO             OFFICE OF MANAGEMENT AND BUDGET
BT    ARCHITECTL & TRANS BARRIER COMPLIANCE BD
CC                  COMMISSION ON CIVIL RIGHTS
CE           

## Accessions
Records of federal employees who joined government. One row = one hiring event.

In [45]:
code_cols = [c for c in a_df.columns if c.endswith("_code")]
print(f"Shape: {a_df.shape[0]:,} rows x {a_df.shape[1]} columns")
print(f"\nCode columns ({len(code_cols)}):")
print(code_cols)
print(f"\nAll columns:")
print(a_df.dtypes.to_string())

Shape: 10,998 rows x 67 columns

Code columns (25):
['accession_category_code', 'agency_code', 'agency_subelement_code', 'appointment_type_code', 'bargaining_unit_code', 'consolidated_statistical_area_code', 'core_based_statistical_area_code', 'duty_station_code', 'duty_station_country_code', 'duty_station_county_code', 'duty_station_state_country_territory_code', 'education_level_code', 'flsa_category_code', 'locality_pay_area_code', 'occupational_category_code', 'occupational_group_code', 'occupational_series_code', 'pay_basis_code', 'pay_plan_code', 'personnel_office_identifier_code', 'position_occupied_code', 'step_or_rate_type_code', 'supervisory_status_code', 'tenure_code', 'work_schedule_code']

All columns:
accession_category                            object
accession_category_code                       object
age_bracket                                   object
agency                                        object
agency_code                                   object
agency_sub

In [46]:
# Sample — working with code columns
a_df[[c for c in a_df.columns if c.endswith("_code")][:6]].head(5)

,accession_category_code,agency_code,agency_subelement_code,appointment_type_code,bargaining_unit_code,consolidated_statistical_area_code
0,AC,AG,AG03,15,7777,NaN
1,AA,AR,ARCE,15,REDACTED,REDACTED
2,AC,AR,ARCE,20,REDACTED,REDACTED
3,AA,AR,ARCH,15,REDACTED,REDACTED
4,AC,AF,AF0M,10,REDACTED,REDACTED


In [47]:
# Key lookup tables for accessions
for key in ["accession_category_code", "agency_code", "tenure_code", "work_schedule_code", "education_level_code", "pay_plan_code", "appointment_type_code"]:
    if key in a_lookups:
        print(f"=== {key} ===")
        print(pd.Series(a_lookups[key]).sort_index().to_string())
        print()
    else:
        print(f"=== {key} === NOT FOUND in lookups")

=== accession_category_code ===
AA                 TRANSFER IN - INDIVIDUAL TRANSFER
AB                       TRANSFER IN - MASS TRANSFER
AC        NEW HIRE - COMPETITIVE SERVICE APPOINTMENT
AD           NEW HIRE - EXCEPTED SERVICE APPOINTMENT
AE    NEW HIRE - SENIOR EXECUTIVE SERVICE (SES) APPT

=== agency_code ===
AB        AMERICAN BATTLE MONUMENTS COMMISSION
AF                 DEPARTMENT OF THE AIR FORCE
AG                   DEPARTMENT OF AGRICULTURE
AH       NAT FOUNDATION ON ARTS AND HUMANITIES
AM           U.S. AGENCY FOR INTERNATIONAL DEV
AR                      DEPARTMENT OF THE ARMY
BF     DEFENSE NUCLEAR FACILITIES SAFETY BOARD
BG        PENSION BENEFIT GUARANTY CORPORATION
BO             OFFICE OF MANAGEMENT AND BUDGET
CM                      DEPARTMENT OF COMMERCE
CT        COMMODITY FUTURES TRADING COMMISSION
DD                       DEPARTMENT OF DEFENSE
DJ                       DEPARTMENT OF JUSTICE
DL                         DEPARTMENT OF LABOR
DN                      

## Employment
A snapshot of the full federal workforce at a point in time. One row = one active employee record.

In [42]:
code_cols = [c for c in e_df.columns if c.endswith("_code")]
print(f"Shape: {e_df.shape[0]:,} rows x {e_df.shape[1]} columns")
print(f"\nCode columns ({len(code_cols)}):")
print(code_cols)
print(f"\nAll columns:")
print(e_df.dtypes.to_string())

Shape: 2,028,138 rows x 64 columns

Code columns (24):
['agency_code', 'agency_subelement_code', 'appointment_type_code', 'bargaining_unit_code', 'consolidated_statistical_area_code', 'core_based_statistical_area_code', 'duty_station_code', 'duty_station_country_code', 'duty_station_county_code', 'duty_station_state_country_territory_code', 'education_level_code', 'flsa_category_code', 'locality_pay_area_code', 'occupational_category_code', 'occupational_group_code', 'occupational_series_code', 'pay_basis_code', 'pay_plan_code', 'personnel_office_identifier_code', 'position_occupied_code', 'step_or_rate_type_code', 'supervisory_status_code', 'tenure_code', 'work_schedule_code']

All columns:
age_bracket                                   object
agency                                        object
agency_code                                   object
agency_subelement                             object
agency_subelement_code                        object
annualized_adjusted_basic_pay     

In [43]:
# Sample — working with code columns
e_df[[c for c in e_df.columns if c.endswith("_code")][:6]].head(5)

,agency_code,agency_subelement_code,appointment_type_code,bargaining_unit_code,consolidated_statistical_area_code,core_based_statistical_area_code
0,DJ,DJ03,10,REDACTED,REDACTED,REDACTED
1,FW,FW00,30,8888,548,47900
2,DJ,DJ09,15,REDACTED,REDACTED,REDACTED
3,HS,HSBC,38,REDACTED,REDACTED,REDACTED
4,DJ,DJ09,10,REDACTED,REDACTED,REDACTED


In [44]:
# Key lookup tables for employment
for key in ["agency_code", "pay_plan_code", "tenure_code", "work_schedule_code", "education_level_code", "occupational_category_code", "supervisory_status_code"]:
    if key in e_lookups:
        print(f"=== {key} ===")
        print(pd.Series(e_lookups[key]).sort_index().to_string())
        print()

=== agency_code ===
AA       ADMIN CONFERENCE OF THE UNITED STATES
AB        AMERICAN BATTLE MONUMENTS COMMISSION
AF                 DEPARTMENT OF THE AIR FORCE
AG                   DEPARTMENT OF AGRICULTURE
AH       NAT FOUNDATION ON ARTS AND HUMANITIES
AM           U.S. AGENCY FOR INTERNATIONAL DEV
AN              AFRICAN DEVELOPMENT FOUNDATION
AP             APPALACHIAN REGIONAL COMMISSION
AR                      DEPARTMENT OF THE ARMY
AU           FEDERAL LABOR RELATIONS AUTHORITY
AW                  ARCTIC RESEARCH COMMISSION
BD              MERIT SYSTEMS PROTECTION BOARD
BF     DEFENSE NUCLEAR FACILITIES SAFETY BOARD
BG        PENSION BENEFIT GUARANTY CORPORATION
BH    CMSN FOR PRES OF AMERICA'S HERITAGE ABRD
BK     JAMES MADISON MEMORIAL FELLOWSHIP FOUND
BO             OFFICE OF MANAGEMENT AND BUDGET
BT    ARCHITECTL & TRANS BARRIER COMPLIANCE BD
BW        NUCLEAR WASTE TECHNICAL REVIEW BOARD
CC                  COMMISSION ON CIVIL RIGHTS
CE                COUNCIL OF ECONOMIC AD